In [0]:
from pyspark.sql.functions import when, col,concat_ws, substring, regexp_replace, sha2, concat, upper, lit

In [0]:
df = spark.read.csv(
    "s3://s3-lge-he-inbound-eic-dev/HEDS/1406/1406_SET Serial Number List.csv",
    header=True,
    inferSchema=True
)
display(df)

In [0]:
df = df.withColumn("LOT_MAC", upper(col("LOT_MAC")))
display(df)

In [0]:
df = df.withColumn(
    "mac_addr",
    concat_ws(":",
        substring(col("LOT_MAC"), 1, 2),
        substring(col("LOT_MAC"), 3, 2),
        substring(col("LOT_MAC"), 5, 2),
        substring(col("LOT_MAC"), 7, 2),
        substring(col("LOT_MAC"), 9, 2),
        substring(col("LOT_MAC"), 11, 2),
    )
)

display(df)

In [0]:
tv_salt = dbutils.secrets.get("admin", "salt")

df=df.withColumn("mac_addr_hashed", when(col("mac_addr").isNull() | (col("mac_addr") == ''), col("mac_addr")).otherwise(sha2(concat(col("mac_addr"), lit(tv_salt)), 256)))

display(df)

In [0]:
df.toDF(*[c.replace(" ", "_") for c in df.columns]).write.format("delta").mode("overwrite").saveAsTable("sandbox.z_yeswook_kim.heds_1406")